# Heart Disease Dataset — EDA
**Owner: Rabiya Tahir**

Complete this notebook before implementing `src/preprocessing/heart_disease.py`.
Goal: understand the data so preprocessing decisions are informed.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/heart_disease_data.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# Data types and missing values
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Target distribution
df['target'].value_counts().plot(kind='bar', color=['steelblue', 'salmon'])
plt.title('Target Distribution (0=No Disease, 1=Disease)')
plt.xlabel('Target')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Feature distributions
df.hist(figsize=(14, 10), bins=20, color='steelblue', edgecolor='white')
plt.suptitle('Feature Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Outlier detection using boxplots
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'target']

fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(col)
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
plt.suptitle('Outlier Detection (Boxplots)', fontsize=14)
plt.tight_layout()
plt.show()

## TODO (Rabiya)
After exploring, answer these questions before writing the preprocessor:
1. Which columns have missing values? How will you handle them?
2. Are there any categorical columns that need encoding?
3. Which columns have strong outliers that need treatment?
4. Does the class distribution need balancing (check target counts above)?

## Rabiya's EDA Answers

**1. Missing values?**
`ca` has 4 missing values and `thal` has 2 — total 6 out of 303 rows.
Strategy: fill with **column median** (both are ordinal/numeric, median is robust to outliers and keeps all 303 rows).

**2. Categorical columns needing encoding?**
None. Every column in the UCI Cleveland dataset is already numeric:
- `sex`: 0/1 binary
- `cp`, `restecg`, `slope`, `ca`, `thal`: integer-coded ordinal/categorical
- `fbs`, `exang`: 0/1 binary
No `pd.get_dummies` or `LabelEncoder` needed.

**3. Strong outliers?**
- `chol`: several values >400 mg/dL (clinically possible but extreme)
- `trestbps`: a few values >180 mmHg
- `oldpeak`: right-skewed, values up to 6.2
Strategy: keep all values (they are clinically valid, not data errors). `StandardScaler` reduces their relative magnitude without discarding real cases.

**4. Class balance?**
164 no-disease (54%) vs 139 disease (46%) — roughly balanced.
No resampling (SMOTE / class_weight) needed; the ~8% imbalance is small enough that all four models will train reliably.